In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
import os
import shutil
from sklearn.model_selection import train_test_split

BASE_PATH = "/kaggle/input/datasets/kumaresanmanickavelu/lyft-udacity-challenge"
OUTPUT_PATH = "/kaggle/working/lyft_split"

# Create output structure
for split in ["train", "val", "test"]:
    os.makedirs(os.path.join(OUTPUT_PATH, split, "images"), exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_PATH, split, "masks"), exist_ok=True)

all_images = []
all_masks = []

# Loop through dataA → dataE
for folder in ["dataA", "dataB", "dataC", "dataD", "dataE"]:
    rgb_path = os.path.join(BASE_PATH, folder, folder, "CameraRGB")
    seg_path = os.path.join(BASE_PATH, folder, folder, "CameraSeg")

    if not os.path.exists(rgb_path):
        continue

    files = sorted(os.listdir(rgb_path))

    for f in files:
        all_images.append(os.path.join(rgb_path, f))
        all_masks.append(os.path.join(seg_path, f))

# Split 70/15/15
train_imgs, temp_imgs, train_masks, temp_masks = train_test_split(
    all_images, all_masks, test_size=0.30, random_state=42
)

val_imgs, test_imgs, val_masks, test_masks = train_test_split(
    temp_imgs, temp_masks, test_size=0.50, random_state=42
)

# Copy function
def move_pairs(imgs, masks, split):
    for img, mask in zip(imgs, masks):
        name = os.path.basename(img)

        shutil.copy(img, os.path.join(OUTPUT_PATH, split, "images", name))
        shutil.copy(mask, os.path.join(OUTPUT_PATH, split, "masks", name))

# Execute
move_pairs(train_imgs, train_masks, "train")
move_pairs(val_imgs, val_masks, "val")
move_pairs(test_imgs, test_masks, "test")

print("✅ Lyft segmentation dataset split done!")

✅ Lyft segmentation dataset split done!


In [22]:
import os
import torch
import numpy as np
from PIL import Image
from torch.utils.data import Dataset
import torchvision.transforms as T

# =========================
# DATASET
# =========================
class SegDataset(Dataset):
    def __init__(self, root, image_size=256):
        self.img_dir = os.path.join(root, "images")
        self.mask_dir = os.path.join(root, "masks")
        self.images = sorted(os.listdir(self.img_dir))

        self.image_size = image_size

        self.transform = T.Compose([
            T.Resize((image_size, image_size)),
            T.ToTensor(),
            T.Normalize([0.485, 0.456, 0.406],
                        [0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):   # ✅ FIXED INDENTATION
        img_path = os.path.join(self.img_dir, self.images[idx])
        mask_path = os.path.join(self.mask_dir, self.images[idx])

        image = Image.open(img_path).convert("RGB")

        # ✅ Correct way (NO RGB conversion)
        mask = Image.open(mask_path)

        image = self.transform(image)

        # ✅ CRITICAL: preserve labels
        mask = mask.resize((self.image_size, self.image_size), Image.NEAREST)

        # convert to tensor
        mask = torch.from_numpy(np.array(mask)).long()

        return image, mask

In [23]:
from torch.utils.data import DataLoader

train_loader = DataLoader(SegDataset("/kaggle/working/lyft_split/train"), batch_size=2)

images, masks = next(iter(train_loader))
print("Unique classes:", torch.unique(masks))

Unique classes: tensor([ 0,  1,  2,  3,  5,  6,  7,  8,  9, 10, 11, 12])


In [29]:
# =========================
# IMPORTS
# =========================
import os
import torch
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision
import torchvision.transforms as T
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm

# =========================
# CONFIG
# =========================
DEVICE = "cuda"
NUM_CLASSES = 13
IMAGE_SIZE = 256
BATCH_SIZE = 8
EPOCHS = 60
LR = 1e-4
PATIENCE = 7

TRAIN_DIR = "/kaggle/working/lyft_split/train"
VAL_DIR = "/kaggle/working/lyft_split/val"

SAVE_PATH = "/kaggle/working/deeplabv3_best.pth"

# =========================
# DATASET
# =========================
class SegDataset(Dataset):
    def __init__(self, root, augment=False):
        self.img_dir = os.path.join(root, "images")
        self.mask_dir = os.path.join(root, "masks")
        self.images = sorted(os.listdir(self.img_dir))
        self.augment = augment

        self.transform = T.Compose([
            T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
            T.ToTensor(),
            T.Normalize([0.485, 0.456, 0.406],
                        [0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.images[idx])
        mask_path = os.path.join(self.mask_dir, self.images[idx])

        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path)

        # 🔥 FIX MASK
        mask = np.array(mask)
        if len(mask.shape) == 3:
            mask = mask[:, :, 0]

        # augmentation
        if self.augment and np.random.rand() > 0.5:
            image = image.transpose(Image.FLIP_LEFT_RIGHT)
            mask = np.fliplr(mask)

        image = self.transform(image)

        mask = Image.fromarray(mask).resize((IMAGE_SIZE, IMAGE_SIZE), Image.NEAREST)
        mask = torch.from_numpy(np.array(mask)).long()

        return image, mask

# =========================
# METRICS
# =========================
def compute_metrics(preds, labels):
    preds = preds.view(-1)
    labels = labels.view(-1)

    acc = (preds == labels).sum().item() / labels.numel()

    ious = []
    for cls in range(NUM_CLASSES):
        pred_i = preds == cls
        label_i = labels == cls

        inter = (pred_i & label_i).sum().item()
        union = (pred_i | label_i).sum().item()

        if union == 0:
            continue
        ious.append(inter / union)

    return acc, np.mean(ious)

# =========================
# DATA LOADERS
# =========================
train_loader = DataLoader(
    SegDataset(TRAIN_DIR, augment=True),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    SegDataset(VAL_DIR),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# =========================
# MODEL (🔥 PRETRAINED)
# =========================
model = torchvision.models.segmentation.deeplabv3_resnet50(
    weights=None,
    weights_backbone="DEFAULT"
)

model.classifier[4] = nn.Conv2d(256, NUM_CLASSES, 1)
model = model.to(DEVICE)

# =========================
# LOSS + OPTIMIZER
# =========================
class_weights = torch.tensor([
    1.0, 2.0, 2.0, 2.0, 3.0, 2.0, 2.0,
    1.0, 1.0, 2.0, 3.0, 2.0, 2.0
]).to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = optim.AdamW(model.parameters(), lr=LR)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=3, factor=0.5
)

# =========================
# AMP (FAST TRAINING)
# =========================
scaler = torch.amp.GradScaler()

# =========================
# TRAIN LOOP
# =========================
best_loss = float("inf")
patience_counter = 0

for epoch in range(EPOCHS):

    # ===== TRAIN =====
    model.train()
    train_loss, train_acc, train_iou = 0, [], []

    for images, masks in tqdm(train_loader):
        images, masks = images.to(DEVICE), masks.to(DEVICE)

        optimizer.zero_grad()

        with torch.amp.autocast(device_type='cuda'):
            outputs = model(images)['out']
            loss = criterion(outputs, masks)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()

        preds = torch.argmax(outputs, dim=1)
        acc, iou = compute_metrics(preds, masks)

        train_acc.append(acc)
        train_iou.append(iou)

    # ===== VALIDATION =====
    model.eval()
    val_loss, val_acc, val_iou = 0, [], []

    with torch.no_grad():
        for images, masks in val_loader:
            images, masks = images.to(DEVICE), masks.to(DEVICE)

            with torch.amp.autocast(device_type='cuda'):
                outputs = model(images)['out']
                loss = criterion(outputs, masks)

            val_loss += loss.item()

            preds = torch.argmax(outputs, dim=1)
            acc, iou = compute_metrics(preds, masks)

            val_acc.append(acc)
            val_iou.append(iou)

    # averages
    train_loss /= len(train_loader)
    val_loss /= len(val_loader)

    print(f"\nEpoch [{epoch+1}/{EPOCHS}]")
    print(f"Train -> Loss: {train_loss:.4f}, Acc: {np.mean(train_acc):.4f}, mIoU: {np.mean(train_iou):.4f}")
    print(f"Val   -> Loss: {val_loss:.4f}, Acc: {np.mean(val_acc):.4f}, mIoU: {np.mean(val_iou):.4f}")

    # scheduler
    scheduler.step(val_loss)

    # early stopping
    if val_loss < best_loss:
        best_loss = val_loss
        torch.save(model.state_dict(), SAVE_PATH)
        print("✅ Best model saved")
        patience_counter = 0
    else:
        patience_counter += 1
        print(f"⏳ Patience: {patience_counter}/{PATIENCE}")

        if patience_counter >= PATIENCE:
            print("⛔ Early stopping triggered")
            break

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 193MB/s] 
100%|██████████| 438/438 [01:26<00:00,  5.05it/s]



Epoch [1/60]
Train -> Loss: 0.3668, Acc: 0.8878, mIoU: 0.4616
Val   -> Loss: 0.2081, Acc: 0.9208, mIoU: 0.5413
✅ Best model saved


100%|██████████| 438/438 [01:21<00:00,  5.39it/s]



Epoch [2/60]
Train -> Loss: 0.1866, Acc: 0.9277, mIoU: 0.5781
Val   -> Loss: 0.1636, Acc: 0.9316, mIoU: 0.5921
✅ Best model saved


...


Epoch [58/60]
Train -> Loss: 0.0948, Acc: 0.9544, mIoU: 0.7189
Val   -> Loss: 0.1031, Acc: 0.9521, mIoU: 0.6930
⏳ Patience: 3/7


100%|██████████| 438/438 [01:22<00:00,  5.32it/s]



Epoch [59/60]
Train -> Loss: 0.0947, Acc: 0.9544, mIoU: 0.7205
Val   -> Loss: 0.1033, Acc: 0.9522, mIoU: 0.6909
⏳ Patience: 4/7


100%|██████████| 438/438 [01:22<00:00,  5.31it/s]



Epoch [60/60]
Train -> Loss: 0.0944, Acc: 0.9545, mIoU: 0.7191
Val   -> Loss: 0.1030, Acc: 0.9523, mIoU: 0.6925
⏳ Patience: 5/7


In [30]:
# =========================
# IMPORTS
# =========================
import os
import torch
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision
import torchvision.transforms as T
import torch.nn as nn
from tqdm import tqdm

# =========================
# CONFIG
# =========================
DEVICE = "cuda"
NUM_CLASSES = 13
IMAGE_SIZE = 256

MODEL_PATH = "/kaggle/working/deeplabv3_best.pth"
TEST_DIR = "/kaggle/working/lyft_split/test"
SAVE_DIR = "/kaggle/working/predictions"

os.makedirs(SAVE_DIR, exist_ok=True)

# =========================
# DATASET
# =========================
class SegDataset(Dataset):
    def __init__(self, root):
        self.img_dir = os.path.join(root, "images")
        self.mask_dir = os.path.join(root, "masks")
        self.images = sorted(os.listdir(self.img_dir))

        self.transform = T.Compose([
            T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
            T.ToTensor(),
            T.Normalize([0.485, 0.456, 0.406],
                        [0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.images[idx])
        mask_path = os.path.join(self.mask_dir, self.images[idx])

        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path)

        image = self.transform(image)

        # FIX MASK
        mask = np.array(mask)
        if len(mask.shape) == 3:
            mask = mask[:, :, 0]

        mask = Image.fromarray(mask).resize((IMAGE_SIZE, IMAGE_SIZE), Image.NEAREST)
        mask = torch.from_numpy(np.array(mask)).long()

        return image, mask, self.images[idx]

# =========================
# METRICS
# =========================
def compute_metrics(preds, labels):
    preds = preds.view(-1)
    labels = labels.view(-1)

    acc = (preds == labels).sum().item() / labels.numel()

    ious = []
    for cls in range(NUM_CLASSES):
        pred_i = preds == cls
        label_i = labels == cls

        inter = (pred_i & label_i).sum().item()
        union = (pred_i | label_i).sum().item()

        if union == 0:
            continue
        ious.append(inter / union)

    return acc, np.mean(ious)

# =========================
# MODEL
# =========================
model = torchvision.models.segmentation.deeplabv3_resnet50(
    weights=None,
    weights_backbone=None
)

model.classifier[4] = nn.Conv2d(256, NUM_CLASSES, 1)
model.load_state_dict(torch.load(MODEL_PATH))
model = model.to(DEVICE)
model.eval()

# =========================
# DATALOADER
# =========================
test_loader = DataLoader(
    SegDataset(TEST_DIR),
    batch_size=4,
    shuffle=False,
    num_workers=2
)

# =========================
# LOSS
# =========================
criterion = nn.CrossEntropyLoss()

# =========================
# COLOR MAP (for visualization)
# =========================
COLORS = np.array([
    [0,0,0],[70,70,70],[190,153,153],[72,0,90],[220,20,60],
    [153,153,153],[157,234,50],[128,64,128],[244,35,232],
    [107,142,35],[0,0,142],[102,102,156],[220,220,0]
])

def decode_mask(mask):
    h, w = mask.shape
    rgb = np.zeros((h, w, 3), dtype=np.uint8)
    for cls in range(NUM_CLASSES):
        rgb[mask == cls] = COLORS[cls]
    return rgb

# =========================
# TEST LOOP
# =========================
test_loss = 0
test_acc, test_iou = [], []

with torch.no_grad():
    for images, masks, names in tqdm(test_loader):

        images = images.to(DEVICE)
        masks = masks.to(DEVICE)

        outputs = model(images)['out']
        loss = criterion(outputs, masks)
        test_loss += loss.item()

        preds = torch.argmax(outputs, dim=1)

        acc, iou = compute_metrics(preds, masks)
        test_acc.append(acc)
        test_iou.append(iou)

        # SAVE PREDICTIONS
        preds = preds.cpu().numpy()

        for i in range(preds.shape[0]):
            colored = decode_mask(preds[i])
            save_path = os.path.join(SAVE_DIR, names[i])
            Image.fromarray(colored).save(save_path)

# =========================
# FINAL METRICS
# =========================
test_loss /= len(test_loader)

print("\n🎯 FINAL TEST RESULTS")
print(f"Loss: {test_loss:.4f}")
print(f"Accuracy: {np.mean(test_acc):.4f}")
print(f"mIoU: {np.mean(test_iou):.4f}")

print(f"\n✅ Predictions saved at: {SAVE_DIR}")

100%|██████████| 188/188 [00:26<00:00,  7.04it/s]


🎯 FINAL TEST RESULTS
Loss: 0.1103
Accuracy: 0.9523
mIoU: 0.6808

✅ Predictions saved at: /kaggle/working/predictions
